# LeetCode 4068 — Maximize Meeting Earnings with Idle Gaps (Hard)

Bài toán LeetCode:

```python
class Solution:
    def maxEarnings(self, meetings: List[List[int]]) -> int:
        ...
```

Link tra cứu: https://leetcode.com/problemset/?search=4068 (Weekly Contest 521 — câu 4, rating khoảng 2189).

Đây là biến thể của bài kinh điển **Weighted Interval Scheduling** (chọn các cuộc họp không chồng nhau, tổng tiền lớn nhất — giống *Maximum Profit in Job Scheduling*, LeetCode 1235). Điểm mới: **thời gian rảnh giữa hai cuộc họp liên tiếp cũng được trả tiền**. Điều đó làm công thức DP có thêm một hạng tử phụ thuộc vào cả hai cuộc họp — và ta sẽ học mẹo **tách hạng tử** để vẫn dùng được **max tiền tố + tìm kiếm nhị phân**.

---

## Sau notebook này, bạn sẽ nắm được

1. Đoạn nửa mở `[start, end)`, điều kiện không chồng nhau.
2. Brute Force thử mọi tập con.
3. DP "cuộc họp cuối cùng là `i`" `O(n²)`.
4. Mẹo tách `f[j] + rev_i + (start_i - end_j)` thành phần chỉ phụ thuộc `i` và phần chỉ phụ thuộc `j`.
5. Sắp theo `end`, `bisect` tìm tiền tố hợp lệ, mảng max tiền tố → `O(n log n)`.
6. Góc nhìn khác: tổng tiền = `Σ(rev - độ dài) + end_cuối - start_đầu`.
7. Lỗi hay gặp, mẹo nhận diện, biến thể và bài tập.

# 1. Đề bài yêu cầu gì?

Cho `meetings[i] = [start_i, end_i, revenue_i]`. Cuộc họp chiếm khoảng **nửa mở** `[start, end)` — hai cuộc họp chỉ chạm nhau ở đầu mút **không** chồng nhau.

Chọn một tập **khác rỗng** các cuộc họp **không chồng nhau**. Tiền nhận được:

- tổng `revenue` của các cuộc họp được chọn, **cộng**
- sắp các cuộc họp được chọn theo thời gian; với mỗi **cặp liên tiếp**, nhận thêm **1 đơn vị cho mỗi đơn vị thời gian rảnh** = `start` của cuộc sau − `end` của cuộc trước.

Thời gian rảnh trước cuộc đầu tiên và sau cuộc cuối cùng không được tính. Trả về tổng tiền lớn nhất.

---

## Ví dụ 1

```python
meetings = [[2,5,4],[6,8,3]]   ->   8
```

Chọn cả hai: `4 + 3 = 7`, rảnh từ 5 tới 6 thêm `1` → `8`.

## Ví dụ 2

```python
meetings = [[3,5,4],[4,7,8],[8,10,3]]   ->   12
```

Chọn cuộc 1 và 2: `8 + 3 = 11`, rảnh từ 7 tới 8 thêm `1` → `12`. (Cuộc 0 chồng với cuộc 1.)

## Ví dụ 3

```python
meetings = [[1,2,2],[4,5,2],[7,9,3]]   ->   11
```

Chọn cả ba: `2 + 2 + 3 = 7`, rảnh `2` (từ 2 tới 4) và `2` (từ 5 tới 7) → `11`.

---

## Ràng buộc

```text
1 <= meetings.length <= 10^5
0 <= start_i < end_i <= 10^9
1 <= revenue_i <= 10^9
```

# 2. Kiến thức nền tảng

## 2.1. Đoạn nửa mở và điều kiện không chồng nhau

`[s1, e1)` và `[s2, e2)` với `e1 <= s2` (cuộc 1 trước cuộc 2) là **không chồng nhau** — kể cả khi `e1 == s2`.

## 2.2. Sắp xếp theo khóa và `bisect`

- `sorted(meetings, key=lambda m: m[1])` sắp theo `end`.
- Trong mảng `ends` đã sắp, `bisect_right(ends, x)` = **số phần tử `<= x`** — tức là số cuộc họp kết thúc trước hoặc đúng lúc `x`.

In [1]:
from bisect import bisect_right
from typing import List

meetings = [[3, 5, 4], [4, 7, 8], [8, 10, 3]]
ms = sorted(meetings, key=lambda m: m[1])
ends = [m[1] for m in ms]
print("ends =", ends)
for x in [4, 5, 8]:
    print(f"số cuộc kết thúc <= {x}: {bisect_right(ends, x)}")

ends = [5, 7, 10]
số cuộc kết thúc <= 4: 0
số cuộc kết thúc <= 5: 1
số cuộc kết thúc <= 8: 2


## 2.3. Mảng max tiền tố

`pre[k] = max(a[0..k-1])` — tính một lần `O(n)`, sau đó hỏi "giá trị lớn nhất trong `k` phần tử đầu" trong `O(1)`.

# 3. Cách 1 — Brute Force: thử mọi tập con

In [2]:
from itertools import combinations


def earnings_of(chosen) -> int:
    chosen = sorted(chosen, key=lambda m: m[0])
    total = sum(m[2] for m in chosen)
    total += sum(chosen[k + 1][0] - chosen[k][1] for k in range(len(chosen) - 1))
    return total


def max_earnings_bruteforce(meetings: List[List[int]]) -> int:
    best = None
    for k in range(1, len(meetings) + 1):
        for chosen in combinations(meetings, k):
            ordered = sorted(chosen, key=lambda m: m[0])
            if all(ordered[t][1] <= ordered[t + 1][0] for t in range(len(ordered) - 1)):
                v = earnings_of(chosen)
                best = v if best is None else max(best, v)
    return best


print(max_earnings_bruteforce([[2, 5, 4], [6, 8, 3]]))                 # 8
print(max_earnings_bruteforce([[3, 5, 4], [4, 7, 8], [8, 10, 3]]))     # 12
print(max_earnings_bruteforce([[1, 2, 2], [4, 5, 2], [7, 9, 3]]))      # 11

8
12
11


`2^n` tập con — chỉ dùng cho `n` nhỏ.

---

# 4. Cách 2 — DP "cuộc họp cuối cùng là `i`": O(n²)

Sắp các cuộc họp theo `end`. Gọi:

```text
f[i] = tiền lớn nhất của một lịch hợp lệ mà cuộc họp CUỐI CÙNG là i
```

- Lịch chỉ có mình `i`: `rev_i`.
- Hoặc cuộc họp ngay trước `i` là `j` với `end_j <= start_i`: phần trước `j` là một lịch tốt nhất kết thúc ở `j` (`f[j]`), cộng `rev_i` và tiền rảnh `start_i - end_j`.

```text
f[i] = max( rev_i ,  max_{j: end_j <= start_i} ( f[j] + rev_i + start_i - end_j ) )
answer = max f[i]
```

Vì sao "lịch tốt nhất kết thúc ở `j`" là đủ? Tiền rảnh giữa `j` và `i` chỉ phụ thuộc `end_j` và `start_i`, không phụ thuộc phần trước `j`. Nên phần trước càng nhiều tiền càng tốt.

In [3]:
def max_earnings_quadratic(meetings: List[List[int]]) -> int:
    ms = sorted(meetings, key=lambda m: m[1])
    n = len(ms)
    f = [0] * n
    for i, (s, e, r) in enumerate(ms):
        f[i] = r
        for j in range(i):
            if ms[j][1] <= s:
                f[i] = max(f[i], f[j] + r + s - ms[j][1])
    return max(f)


print(max_earnings_quadratic([[2, 5, 4], [6, 8, 3]]))
print(max_earnings_quadratic([[3, 5, 4], [4, 7, 8], [8, 10, 3]]))
print(max_earnings_quadratic([[1, 2, 2], [4, 5, 2], [7, 9, 3]]))

8
12
11


`O(n²)` với `n = 10^5` là `10^10` — quá chậm.

# 5. Bước đột phá: tách hạng tử, max tiền tố, tìm kiếm nhị phân

Viết lại:

```text
f[j] + rev_i + start_i - end_j  =  (rev_i + start_i)  +  (f[j] - end_j)
                                    \_____ chỉ i ____/    \_ chỉ j _/
```

Vậy:

```text
f[i] = max( rev_i ,  rev_i + start_i + max_{j hợp lệ} (f[j] - end_j) )
```

Hai quan sát:

1. Đã sắp theo `end`, nên các `j` có `end_j <= start_i` là **một tiền tố** `ms[0..k-1]`, với `k = bisect_right(ends, start_i)`. (Tiền tố này luôn nằm trước `i` vì `end_j <= start_i < end_i`.)
2. Giữ mảng `pre[k] = max_{j < k} (f[j] - end_j)` → tra trong `O(1)`.

Mỗi cuộc họp: một `bisect` `O(log n)` + `O(1)` → tổng `O(n log n)`.

## Mô phỏng ví dụ 2

Sắp theo `end`: `[3,5,4]`, `[4,7,8]`, `[8,10,3]`; `ends = [5, 7, 10]`.

| i | (s, e, r) | k = #end ≤ s | pre[k] | f[i] | f[i] − e | pre[i+1] |
|---:|---|---:|---:|---|---:|---:|
| 0 | (3, 5, 4) | 0 | −∞ | 4 (một mình) | −1 | −1 |
| 1 | (4, 7, 8) | 0 | −∞ | 8 (một mình) | 1 | 1 |
| 2 | (8, 10, 3) | 2 | 1 | `max(3, 3 + 8 + 1) = 12` | 2 | 2 |

Đáp án `max f = 12` ✔.

In [4]:
def max_earnings_fast(meetings: List[List[int]], verbose: bool = False) -> int:
    ms = sorted(meetings, key=lambda m: m[1])
    ends = [m[1] for m in ms]
    NEG = float("-inf")
    pre = [NEG] * (len(ms) + 1)            # pre[k] = max(f[j] - end_j) với j < k
    best = NEG
    for i, (s, e, r) in enumerate(ms):
        k = bisect_right(ends, s, 0, i)    # các j < i có end_j <= s
        f = r
        if pre[k] > NEG:
            f = max(f, r + s + pre[k])
        best = max(best, f)
        pre[i + 1] = max(pre[i], f - e)
        if verbose:
            print(f"i = {i}, (s, e, r) = ({s}, {e}, {r}): k = {k}, pre[k] = {pre[k]}, f = {f}, pre[i+1] = {pre[i + 1]}")
    return best


print(max_earnings_fast([[3, 5, 4], [4, 7, 8], [8, 10, 3]], verbose=True))

i = 0, (s, e, r) = (3, 5, 4): k = 0, pre[k] = -inf, f = 4, pre[i+1] = -1
i = 1, (s, e, r) = (4, 7, 8): k = 0, pre[k] = -inf, f = 8, pre[i+1] = 1
i = 2, (s, e, r) = (8, 10, 3): k = 2, pre[k] = 1, f = 12, pre[i+1] = 2
12


# 6. Cách 4 — Góc nhìn khác: tổng tiền theo "đầu" và "cuối"

Với các cuộc họp được chọn `m1, m2, ..., mk` theo thời gian, tổng thời gian rảnh là một **tổng lồng (telescoping)**:

```text
Σ (start_{t+1} - end_t) = (end_k - start_1) - Σ (end_t - start_t)
```

(Khoảng từ đầu cuộc 1 tới cuối cuộc k = tổng thời lượng họp + tổng thời gian rảnh.)

Do đó:

```text
tổng tiền = Σ (rev_t - (end_t - start_t))  +  end_k  -  start_1
```

Đặt `w_t = rev_t - (end_t - start_t)` ("lời ròng" của cuộc họp). Bài toán trở thành: chọn tập không chồng nhau, tối đa `Σ w + end_cuối - start_đầu`. DP:

```text
g[i] = lịch kết thúc ở i, chưa cộng end_i:   g[i] = w_i + max( -start_i ,  max_{j hợp lệ} g[j] )
answer = max ( g[i] + end_i )
```

Cách này cho cùng kết quả, và cho thấy bản chất: tiền rảnh **không phụ thuộc** cách chia nhỏ, chỉ phụ thuộc cuộc đầu và cuộc cuối!

In [5]:
def max_earnings_telescoping(meetings: List[List[int]]) -> int:
    ms = sorted(meetings, key=lambda m: m[1])
    ends = [m[1] for m in ms]
    NEG = float("-inf")
    pre = [NEG] * (len(ms) + 1)             # pre[k] = max g[j], j < k
    best = NEG
    for i, (s, e, r) in enumerate(ms):
        w = r - (e - s)
        k = bisect_right(ends, s, 0, i)
        g = w + max(-s, pre[k])
        best = max(best, g + e)
        pre[i + 1] = max(pre[i], g)
    return best


print(max_earnings_telescoping([[2, 5, 4], [6, 8, 3]]))
print(max_earnings_telescoping([[3, 5, 4], [4, 7, 8], [8, 10, 3]]))
print(max_earnings_telescoping([[1, 2, 2], [4, 5, 2], [7, 9, 3]]))

8
12
11


# 7. Phiên bản đúng format LeetCode

In [6]:
from bisect import bisect_right
from typing import List


class Solution:
    def maxEarnings(self, meetings: List[List[int]]) -> int:
        ms = sorted(meetings, key=lambda m: m[1])
        ends = [m[1] for m in ms]
        NEG = float("-inf")
        pre = [NEG] * (len(ms) + 1)
        best = NEG
        for i, (s, e, r) in enumerate(ms):
            k = bisect_right(ends, s, 0, i)
            f = r
            if pre[k] != NEG and r + s + pre[k] > f:
                f = r + s + pre[k]
            if f > best:
                best = f
            pre[i + 1] = pre[i] if pre[i] > f - e else f - e
        return best

In [7]:
sol = Solution()
tests = [
    ([[2, 5, 4], [6, 8, 3]], 8, "ví dụ 1"),
    ([[3, 5, 4], [4, 7, 8], [8, 10, 3]], 12, "ví dụ 2"),
    ([[1, 2, 2], [4, 5, 2], [7, 9, 3]], 11, "ví dụ 3"),
    ([[0, 1, 5]], 5, "một cuộc họp"),
    ([[0, 5, 1], [5, 9, 1]], 2, "chạm nhau: rảnh 0"),
    ([[0, 10, 100], [1, 2, 1], [8, 9, 1]], 100, "cuộc dài rất lời"),
    ([[0, 1, 1], [100, 101, 1]], 101, "rảnh rất dài"),
]
for m, expected, name in tests:
    result = sol.maxEarnings(m)
    assert result == expected, (name, result, expected)
    print(f"PASS | {name:22} -> {result}")

PASS | ví dụ 1                -> 8
PASS | ví dụ 2                -> 12
PASS | ví dụ 3                -> 11
PASS | một cuộc họp           -> 5
PASS | chạm nhau: rảnh 0      -> 2
PASS | cuộc dài rất lời       -> 100
PASS | rảnh rất dài           -> 101


# 8. Vì sao đúng?

1. **Cấu trúc tối ưu:** trong một lịch tối ưu kết thúc ở `i`, phần trước `i` (kết thúc ở `j`) phải tối ưu trong số các lịch kết thúc ở `j` — vì tiền phụ thêm `rev_i + start_i - end_j` không phụ thuộc phần trước `j`.
2. **Tiền tố hợp lệ:** sắp theo `end`, điều kiện `end_j <= start_i` chọn ra đúng `k = bisect_right(ends, start_i)` phần tử đầu (giới hạn tìm trong `[0, i)` cho chắc chắn `j ≠ i`).
3. **Tách hạng tử:** `max_j (f[j] + rev_i + start_i - end_j) = rev_i + start_i + max_j (f[j] - end_j)` = `rev_i + start_i + pre[k]`.
4. **Đáp án** là `max f[i]` vì lịch khác rỗng tốt nhất kết thúc ở một cuộc họp nào đó.

# 9. Độ phức tạp

| Cách | Ý tưởng | Thời gian | Bộ nhớ |
|---|---|---:|---:|
| 1. Brute Force | mọi tập con | `O(2^n · n log n)` | `O(n)` |
| 2. DP O(n²) | thử mọi `j` trước `i` | `O(n²)` | `O(n)` |
| 3. Tách hạng tử | sắp + `bisect` + max tiền tố | `O(n log n)` | `O(n)` |
| 4. Tổng lồng | `w = rev - độ dài` | `O(n log n)` | `O(n)` |

# 10. Đối chiếu bốn cách

In [8]:
import random
import time

random.seed(4068)
for _ in range(3000):
    m = []
    for _ in range(random.randint(1, 7)):
        s = random.randint(0, 12)
        m.append([s, s + random.randint(1, 5), random.randint(1, 6)])
    expected = max_earnings_bruteforce(m)
    assert max_earnings_quadratic(m) == expected, m
    assert max_earnings_fast(m) == expected, m
    assert max_earnings_telescoping(m) == expected, m
    assert sol.maxEarnings(m) == expected, m
print("3000 bộ test ngẫu nhiên: 5 hàm cho cùng kết quả!")

big = []
for _ in range(10**5):
    s = random.randint(0, 10**9 - 1)
    big.append([s, min(10**9, s + random.randint(1, 10**4)), random.randint(1, 10**9)])
t0 = time.perf_counter()
print("n = 10^5:", sol.maxEarnings(big), f"({time.perf_counter() - t0:.2f}s)")

3000 bộ test ngẫu nhiên: 5 hàm cho cùng kết quả!


n = 10^5: 38011194651950 (0.08s)


# 11. Những lỗi dễ mắc

## Lỗi 1 — Dùng `bisect_left` (coi chạm nhau là chồng nhau)

Đoạn nửa mở: `end_j == start_i` là **hợp lệ**. `bisect_left(ends, s)` đếm số `end < s`, bỏ sót cuộc họp kết thúc đúng lúc `s`.

In [9]:
from bisect import bisect_left


def wrong_bisect_left(meetings):
    ms = sorted(meetings, key=lambda m: m[1])
    ends = [m[1] for m in ms]
    NEG = float("-inf")
    pre = [NEG] * (len(ms) + 1)
    best = NEG
    for i, (s, e, r) in enumerate(ms):
        k = bisect_left(ends, s, 0, i)                      # SAI
        f = r if pre[k] == NEG else max(r, r + s + pre[k])
        best = max(best, f)
        pre[i + 1] = max(pre[i], f - e)
    return best


print("Sai :", wrong_bisect_left([[0, 5, 1], [5, 9, 1]]))
print("Đúng:", sol.maxEarnings([[0, 5, 1], [5, 9, 1]]))

Sai : 1
Đúng: 2


## Lỗi 2 — Cộng cả thời gian rảnh trước cuộc đầu tiên

Nếu khởi tạo "lịch rỗng kết thúc ở thời điểm 0", rồi cộng `start_i - 0` cho cuộc đầu tiên, ta trả tiền cho khoảng rảnh trước cuộc đầu — sai đề.

In [10]:
def wrong_idle_before_first(meetings):
    ms = sorted(meetings, key=lambda m: m[1])
    ends = [m[1] for m in ms]
    pre = [0] * (len(ms) + 1)                # SAI: coi như có "cuộc họp ảo" kết thúc ở 0
    best = float("-inf")
    for i, (s, e, r) in enumerate(ms):
        k = bisect_right(ends, s, 0, i)
        f = r + s + pre[k]
        best = max(best, f)
        pre[i + 1] = max(pre[i], f - e)
    return best


print("Sai :", wrong_idle_before_first([[2, 5, 4], [6, 8, 3]]))
print("Đúng:", sol.maxEarnings([[2, 5, 4], [6, 8, 3]]))

Sai : 10
Đúng: 8


## Lỗi 3 — Sắp theo `start` rồi tìm "tiền tố hợp lệ"

Khi sắp theo `start`, các cuộc `j` có `end_j <= start_i` **không** tạo thành tiền tố (một cuộc bắt đầu sớm có thể kết thúc rất muộn). Phải sắp theo `end`.

## Lỗi 4 — Cho phép tập rỗng (đáp án 0)

Đề yêu cầu tập **khác rỗng**. Vì `revenue >= 1` nên đáp án luôn `>= 1`; khởi tạo `best = -∞` và luôn xét "một mình `i`".

## Lỗi 5 — Không giới hạn `bisect` trong `[0, i)`

Với các cuộc họp có `end` bằng nhau, `bisect_right(ends, s)` (không giới hạn) có thể trả về chỉ số `> i` nếu dữ liệu đặc biệt (ví dụ `start == end` — đề không cho, nhưng giới hạn `hi = i` là thói quen an toàn).

# 12. Mẹo nhận diện

1. Chọn các khoảng **không chồng nhau**, tối đa tổng giá trị → **Weighted Interval Scheduling**: sắp theo `end`, DP "khoảng cuối là `i`", `bisect` tìm khoảng trước.
2. Hạng tử chuyển tiếp phụ thuộc **cả `i` lẫn `j`** nhưng **tách được** thành `A(i) + B(j)` → giữ `max B(j)` theo tiền tố.
3. Tổng các "khoảng hở giữa các phần liên tiếp" → **tổng lồng**: chỉ còn đầu và cuối.

Các bài liên quan: *Maximum Profit in Job Scheduling* (LeetCode 1235), *Maximum Number of Events That Can Be Attended II* (1751), *Two Best Non-Overlapping Events* (2054).

# 13. Biến thể

## Biến thể 1 — Không có tiền rảnh (LeetCode 1235)

Bỏ hạng tử `start_i - end_j`: `f[i] = rev_i + max(0, max_{j hợp lệ} f[j])`.

In [11]:
def job_scheduling(meetings: List[List[int]]) -> int:
    ms = sorted(meetings, key=lambda m: m[1])
    ends = [m[1] for m in ms]
    pre = [0] * (len(ms) + 1)
    for i, (s, e, r) in enumerate(ms):
        k = bisect_right(ends, s, 0, i)
        f = r + pre[k]
        pre[i + 1] = max(pre[i], f)
    return pre[-1]


print(job_scheduling([[1, 3, 50], [2, 4, 10], [3, 5, 40], [3, 6, 70]]))   # 120

120


## Biến thể 2 — Mỗi đơn vị rảnh **bị trừ** `c` đồng

Hạng tử thành `-c·(start_i - end_j) = -c·start_i + c·end_j`, vẫn tách được: `f[i] = max( rev_i ,  rev_i - c·start_i + max_j (f[j] + c·end_j) )` (lịch chỉ có `i` thì không bị trừ).

In [12]:
def earnings_with_penalty(meetings: List[List[int]], c: int) -> int:
    ms = sorted(meetings, key=lambda m: m[1])
    ends = [m[1] for m in ms]
    NEG = float("-inf")
    pre = [NEG] * (len(ms) + 1)
    best = NEG
    for i, (s, e, r) in enumerate(ms):
        k = bisect_right(ends, s, 0, i)
        f = r if pre[k] == NEG else max(r, r - c * s + pre[k])
        best = max(best, f)
        pre[i + 1] = max(pre[i], f + c * e)
    return best


print(earnings_with_penalty([[0, 1, 5], [3, 4, 5]], 1))   # 5 + 5 - 2 = 8
print(earnings_with_penalty([[0, 1, 5], [9, 10, 5]], 1))  # rảnh 8 bị trừ 8 -> chỉ chọn 1 cuộc: 5

8
5


# 14. Template tư duy

```python
ms.sort(key=end); ends = [...]
pre = [-INF] * (n + 1)                   # pre[k] = max B(j), j < k
for i, (s, e, r) in enumerate(ms):
    k = bisect_right(ends, s, 0, i)      # các j kết thúc trước s (nửa mở)
    f = max(một_mình(i), A(i) + pre[k])
    best = max(best, f)
    pre[i + 1] = max(pre[i], B(i, f))
```

# 15. Bài tập vận dụng

## Bài 1 — Tính bằng tay

Ví dụ 3 (`[[1,2,2],[4,5,2],[7,9,3]]`): lập bảng `i | k | pre[k] | f[i] | pre[i+1]`. Kiểm tra bằng cell dưới.

In [13]:
# Kiểm tra Bài 1
max_earnings_fast([[1, 2, 2], [4, 5, 2], [7, 9, 3]], verbose=True)

i = 0, (s, e, r) = (1, 2, 2): k = 0, pre[k] = -inf, f = 2, pre[i+1] = 0
i = 1, (s, e, r) = (4, 5, 2): k = 1, pre[k] = 0, f = 6, pre[i+1] = 1
i = 2, (s, e, r) = (7, 9, 3): k = 2, pre[k] = 1, f = 11, pre[i+1] = 2


11

## Bài 2 — Dự đoán output

```python
ends = [2, 5, 5, 9]
print(bisect_right(ends, 5), bisect_left(ends, 5), bisect_right(ends, 1), bisect_right(ends, 9, 0, 2))
```

In [14]:
ends = [2, 5, 5, 9]
print(bisect_right(ends, 5), bisect_left(ends, 5), bisect_right(ends, 1), bisect_right(ends, 9, 0, 2))

3 1 0 2


## Bài 3 — Hoàn thiện cách tách hạng tử

In [15]:
def exercise_3_fast(meetings: List[List[int]]) -> int:
    # TODO 1: sắp theo end, lập mảng ends
    # TODO 2: với mỗi i: k = bisect_right(ends, s, 0, i); f = max(r, r + s + pre[k])
    # TODO 3: pre[i+1] = max(pre[i], f - e); trả về max f
    raise NotImplementedError("Hãy tự hoàn thiện trước khi xem đáp án.")


# assert exercise_3_fast([[3, 5, 4], [4, 7, 8], [8, 10, 3]]) == 12

## Bài 4 — Maximum Profit in Job Scheduling (LeetCode 1235)

```python
job(startTime=[1,2,3,3], endTime=[3,4,5,6], profit=[50,10,40,70])   -> 120
```

In [16]:
def exercise_4_job(startTime: List[int], endTime: List[int], profit: List[int]) -> int:
    # TODO
    raise NotImplementedError


# assert exercise_4_job([1, 2, 3, 3], [3, 4, 5, 6], [50, 10, 40, 70]) == 120

## Bài 5 — Biến thể: chọn **đúng 2** cuộc họp

Tối đa tiền khi phải chọn **đúng hai** cuộc họp không chồng nhau (tính cả tiền rảnh giữa chúng), hoặc `-1` nếu không thể.

```python
two_meetings([[1,2,2],[4,5,2],[7,9,3]])   -> 2 + 3 + (7 - 2) = 10
```

In [17]:
def exercise_5_two_meetings(meetings: List[List[int]]) -> int:
    # TODO
    raise NotImplementedError


# assert exercise_5_two_meetings([[1, 2, 2], [4, 5, 2], [7, 9, 3]]) == 10

## Bài 6 — Mini challenge: trả về các cuộc họp được chọn

Truy vết DP để trả về danh sách chỉ số (trong input ban đầu) của một lịch tối ưu. Kiểm tra bằng `earnings_of`.

In [18]:
def exercise_6_chosen(meetings: List[List[int]]) -> List[int]:
    # TODO
    raise NotImplementedError


# idx = exercise_6_chosen([[3, 5, 4], [4, 7, 8], [8, 10, 3]])
# assert earnings_of([[3, 5, 4], [4, 7, 8], [8, 10, 3]][k] for k in idx) == 12

## Gợi ý (mở từng mức)

**Bài 3**

- Mức 1: `ms = sorted(meetings, key=lambda m: m[1])`.
- Mức 2: `pre` dài `n + 1`, khởi tạo `-∞`.
- Mức 3: chỉ dùng nhánh nối khi `pre[k] > -∞`.

**Bài 4**

- Mức 1: gộp thành bộ ba `(start, end, profit)`.
- Mức 2: dùng `job_scheduling` ở biến thể 1.
- Mức 3: `pre[i+1] = max(pre[i], profit_i + pre[k])`.

**Bài 5**

- Mức 1: cố định cuộc thứ hai `i`; cuộc thứ nhất `j` có `end_j <= start_i`.
- Mức 2: tiền = `rev_j - end_j` + `rev_i + start_i`.
- Mức 3: max tiền tố của `rev_j - end_j` theo thứ tự `end`.

**Bài 6**

- Mức 1: lưu thêm `best_j[k]` — chỉ số `j` đạt `pre[k]`, và `prev[i]`.
- Mức 2: `prev[i] = best_j[k]` nếu nhánh nối thắng, ngược lại `-1`.
- Mức 3: từ `i` đạt `max f`, lần theo `prev`.

# 16. Đáp án tham khảo

## Đáp án Bài 1

Sắp theo `end`: `(1,2,2)`, `(4,5,2)`, `(7,9,3)`; `ends = [2, 5, 9]`.

| i | k | pre[k] | f[i] | pre[i+1] |
|---:|---:|---:|---|---:|
| 0 | 0 | −∞ | 2 | `2 - 2 = 0` |
| 1 | 1 | 0 | `max(2, 2 + 4 + 0) = 6` | `max(0, 6 - 5) = 1` |
| 2 | 2 | 1 | `max(3, 3 + 7 + 1) = 11` | `max(1, 11 - 9) = 2` |

Đáp án `11` ✔.

## Đáp án Bài 2

```text
3 1 0 2
```

In [19]:
# Đáp án Bài 3
def answer_3_fast(meetings: List[List[int]]) -> int:
    ms = sorted(meetings, key=lambda m: m[1])
    ends = [m[1] for m in ms]
    NEG = float("-inf")
    pre = [NEG] * (len(ms) + 1)
    best = NEG
    for i, (s, e, r) in enumerate(ms):
        k = bisect_right(ends, s, 0, i)
        f = r if pre[k] == NEG else max(r, r + s + pre[k])
        best = max(best, f)
        pre[i + 1] = max(pre[i], f - e)
    return best


assert answer_3_fast([[3, 5, 4], [4, 7, 8], [8, 10, 3]]) == 12
print("Bài 3: PASS")

Bài 3: PASS


In [20]:
# Đáp án Bài 4
def answer_4_job(startTime: List[int], endTime: List[int], profit: List[int]) -> int:
    return job_scheduling([[s, e, p] for s, e, p in zip(startTime, endTime, profit)])


assert answer_4_job([1, 2, 3, 3], [3, 4, 5, 6], [50, 10, 40, 70]) == 120
print("Bài 4: PASS")

Bài 4: PASS


In [21]:
# Đáp án Bài 5
def answer_5_two_meetings(meetings: List[List[int]]) -> int:
    ms = sorted(meetings, key=lambda m: m[1])
    ends = [m[1] for m in ms]
    NEG = float("-inf")
    pre = [NEG] * (len(ms) + 1)
    best = NEG
    for i, (s, e, r) in enumerate(ms):
        k = bisect_right(ends, s, 0, i)
        if pre[k] > NEG:
            best = max(best, r + s + pre[k])
        pre[i + 1] = max(pre[i], r - e)
    return -1 if best == NEG else best


def brute_two(meetings):
    best = -1
    for a, b in combinations(meetings, 2):
        x, y = sorted([a, b], key=lambda m: m[0])
        if x[1] <= y[0]:
            best = max(best, x[2] + y[2] + y[0] - x[1])
    return best


assert answer_5_two_meetings([[1, 2, 2], [4, 5, 2], [7, 9, 3]]) == 10
random.seed(5)
for _ in range(1000):
    m = []
    for _ in range(random.randint(1, 6)):
        s = random.randint(0, 10)
        m.append([s, s + random.randint(1, 4), random.randint(1, 6)])
    assert answer_5_two_meetings(m) == brute_two(m), m
print("Bài 5: PASS (khớp brute force trên 1000 test)")

Bài 5: PASS (khớp brute force trên 1000 test)


In [22]:
# Đáp án Bài 6
def answer_6_chosen(meetings: List[List[int]]) -> List[int]:
    order = sorted(range(len(meetings)), key=lambda t: meetings[t][1])
    ends = [meetings[t][1] for t in order]
    NEG = float("-inf")
    pre, arg = [NEG] * (len(order) + 1), [-1] * (len(order) + 1)
    f, prev = [0] * len(order), [-1] * len(order)
    for i, t in enumerate(order):
        s, e, r = meetings[t]
        k = bisect_right(ends, s, 0, i)
        f[i] = r
        if pre[k] > NEG and r + s + pre[k] > r:
            f[i], prev[i] = r + s + pre[k], arg[k]
        if f[i] - e > pre[i]:
            pre[i + 1], arg[i + 1] = f[i] - e, i
        else:
            pre[i + 1], arg[i + 1] = pre[i], arg[i]
    i = max(range(len(order)), key=lambda x: f[x])
    res = []
    while i != -1:
        res.append(order[i])
        i = prev[i]
    return res[::-1]


ex = [[3, 5, 4], [4, 7, 8], [8, 10, 3]]
idx = answer_6_chosen(ex)
print(idx)
assert earnings_of([ex[k] for k in idx]) == 12
random.seed(6)
for _ in range(1000):
    m = []
    for _ in range(random.randint(1, 7)):
        s = random.randint(0, 12)
        m.append([s, s + random.randint(1, 5), random.randint(1, 6)])
    ids = answer_6_chosen(m)
    chosen = sorted((m[k] for k in ids), key=lambda x: x[0])
    assert all(chosen[t][1] <= chosen[t + 1][0] for t in range(len(chosen) - 1))
    assert earnings_of(chosen) == sol.maxEarnings(m)
print("Bài 6: PASS")

[1, 2]
Bài 6: PASS


# 17. Tổng kết

**Maximize Meeting Earnings with Idle Gaps**:

1. Sắp theo `end`; DP `f[i]` = lịch tốt nhất kết thúc ở `i`.
2. Tách `f[j] + rev_i + start_i - end_j = (rev_i + start_i) + (f[j] - end_j)`.
3. `bisect_right` (nửa mở) tìm tiền tố hợp lệ, `pre[k]` = max `f[j] - end_j`.
4. Góc nhìn tổng lồng: tiền rảnh chỉ phụ thuộc cuộc đầu và cuộc cuối.

## Độ phức tạp

```text
Time:  O(n log n)
Space: O(n)
```

# 18. Checklist trước khi submit

- [ ] Sắp theo **`end`**, không phải `start`?
- [ ] `bisect_right` (chạm nhau là hợp lệ vì nửa mở)?
- [ ] Không trả tiền rảnh trước cuộc đầu / sau cuộc cuối?
- [ ] Luôn xét phương án "chỉ một cuộc họp"?
- [ ] `pre` khởi tạo `-∞`, chỉ nối khi `pre[k] > -∞`?